# AML Benchmark — análise comparativa de desempenho

Compara dois pipelines de streaming sobre o mesmo modelo:
1. **Kafka**: `aml_kafka_producer.ipynb` → tópico → `aml_kafka_consumer.ipynb` → `predictions_kafka/`
2. **File-based**: `aml_producer2.ipynb` → pasta HDFS → `aml_file_consumer.ipynb` → `predictions_file/`

Compara também os dois tamanhos de dataset em modo batch (Small vs Medium).

Métricas:
- **Throughput** (msgs/s) durante a corrida
- **Qualidade do modelo** (accuracy, precision, recall, F1) sobre as previsões com `truth` conhecida
- **Latência** de processamento (kafka_ts → ingest, ou file mtime → ingest)
- **Tempo de leitura batch** das duas tabelas (Small vs Medium)

In [ ]:
import time
from pyspark.sql import SparkSession, functions as F

HDFS_BASE = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project"
PRED_KAFKA = f"{HDFS_BASE}/stream/predictions_kafka"
PRED_FILE  = f"{HDFS_BASE}/stream/predictions_file"
METR_KAFKA = f"{HDFS_BASE}/stream/metrics_kafka"
METR_FILE  = f"{HDFS_BASE}/stream/metrics_file"
DS_SMALL   = f"{HDFS_BASE}/dataset/LI-Small_Trans.csv"
DS_MEDIUM  = f"{HDFS_BASE}/dataset/LI-Medium_Trans.csv"

spark = SparkSession.builder.appName("AML Benchmark").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

## 1) Qualidade do modelo (previsões com `truth` conhecida)

Como as mensagens trazem `Is Laundering` da fonte original, podemos calcular métricas reais
comparando previsão vs verdade. Numa demo "real" o `truth` não viria — usamos só para validação.

In [ ]:
def quality_metrics(path, label):
    try:
        df = spark.read.parquet(path)
    except Exception as e:
        print(f"[{label}] sem dados em {path}: {e}")
        return
    n = df.count()
    if n == 0:
        print(f"[{label}] 0 previsões.")
        return
    m = df.agg(
        F.sum(F.when((F.col("prediction") == 1) & (F.col("truth") == 1), 1).otherwise(0)).alias("tp"),
        F.sum(F.when((F.col("prediction") == 1) & (F.col("truth") == 0), 1).otherwise(0)).alias("fp"),
        F.sum(F.when((F.col("prediction") == 0) & (F.col("truth") == 1), 1).otherwise(0)).alias("fn"),
        F.sum(F.when((F.col("prediction") == 0) & (F.col("truth") == 0), 1).otherwise(0)).alias("tn"),
    ).collect()[0]
    tp, fp, fn, tn = m["tp"] or 0, m["fp"] or 0, m["fn"] or 0, m["tn"] or 0
    acc  = (tp + tn) / n if n else 0
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec  = tp / (tp + fn) if (tp + fn) else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    print(f"[{label}] n={n:,}  TP={tp:,} FP={fp:,} FN={fn:,} TN={tn:,}")
    print(f"         acc={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")

quality_metrics(PRED_KAFKA, "kafka")
quality_metrics(PRED_FILE,  "file ")

## 2) Throughput (msgs/s) — pelas métricas em janelas

In [ ]:
def throughput(path, label):
    try:
        m = spark.read.parquet(path)
    except Exception as e:
        print(f"[{label}] sem métricas em {path}")
        return
    if m.count() == 0:
        print(f"[{label}] sem janelas registadas.")
        return
    summary = m.agg(
        F.avg("n_msgs").alias("avg_per_window"),
        F.max("n_msgs").alias("peak_per_window"),
        F.sum("n_msgs").alias("total"),
        F.count("*").alias("windows"),
    ).collect()[0]
    # janelas são de 30s
    avg_per_sec = (summary["avg_per_window"] or 0) / 30.0
    peak_per_sec = (summary["peak_per_window"] or 0) / 30.0
    print(f"[{label}] janelas={summary['windows']}  total_msgs={summary['total']:,}")
    print(f"         throughput médio={avg_per_sec:.0f} msgs/s  pico={peak_per_sec:.0f} msgs/s")

throughput(METR_KAFKA, "kafka")
throughput(METR_FILE,  "file ")

## 3) Batch read — Small vs Medium

Mede o tempo que o Spark demora a ler+contar cada dataset em batch (referência para comparação).

In [ ]:
def batch_read_time(path, label):
    t0 = time.time()
    n = spark.read.option("header", True).option("inferSchema", False).csv(path).count()
    dt = time.time() - t0
    print(f"[{label}] {n:>12,} linhas em {dt:6.1f}s ({n/dt:,.0f} linhas/s)")
    return n, dt

batch_read_time(DS_SMALL,  "Small ")
batch_read_time(DS_MEDIUM, "Medium")

## 4) Distribuição de classificações (sanity check)

In [ ]:
for path, label in [(PRED_KAFKA, "kafka"), (PRED_FILE, "file")]:
    try:
        df = spark.read.parquet(path)
    except Exception:
        continue
    print(f"\n--- {label} ---")
    df.groupBy("prediction", "truth").count().orderBy("prediction", "truth").show()